#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [18]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_5mm.csv")

cajas_redimensionadas_5mm = pd.read_csv("Redimensionamientos/redimensionamientos_5mm.csv")

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Calculamos también las cajas asignables a cada producto en la lista de productos.

In [19]:
def guardar_cajas_y_productos(grosor):
    #caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
    #                                              on="caja_tipo_id")
    cajas = [
        Caja(
            caja_id = row["caja_id"],
            dim_interior_ancho = row["ancho"],
            dim_interior_largo = row["largo"],
            dim_interior_alto = row["alto"]
        )
        for _, row in cajas_redimensionadas_5mm.iterrows()
    ]

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = [
        Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    ]
    
    # Crear un diccionario de cajas por producto desde el CSV de factibilidad
    cajas_por_producto = {}
    for _, row in factibilidad.iterrows():
        codigo = row['codigo_producto']
        cajas_str = row['cajas_asignables_id']
        
        if isinstance(cajas_str, str) and cajas_str:
            # Separar por '; ' y limpiar
            cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
            cajas_por_producto[codigo] = cajas_list

    # Recorrer la lista de productos en orden y agregar las cajas
    for producto in productos:
        if producto.codigo_producto in cajas_por_producto:
            for caja_id in cajas_por_producto[producto.codigo_producto]:
                producto.agregar_caja_asignable(caja_id)
                
    # Elegir grosor
    for caja in cajas:
        caja.elegir_grosor(grosor_mm=grosor)
    return cajas, productos

#### **Greedy 1: Elección por menor costo**

Empecemos eligiendo un grosor de 5mm para todos los tipos de cajas

In [ ]:
def solucion_greedy(grosor, cajas, productos_ordenados, criterio):

    solucion = Solucion(grosor=grosor)

    for producto in productos_ordenados:
        metricas = {}  # caja_id -> (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
        
        for caja_id in producto.cajas_asignables:
            caja = buscar_caja_por_id(caja_id)
            
            # Volumen requerido actual (antes de asignar este producto)
            volumen_total = caja.unidades_total_requeridas()
            utilizacion_pallet = caja.utilizacion_pallet()
            
            # Costo simulado de asignar este producto a esta caja
            asignacion = Asignacion(producto, caja)
            caja.asignar_producto(producto)
            
            # Calculamos los costos del producto utilizando ese tipo de caja
            costo_packaging = asignacion.costo_packaging_producto_total()
            costo_flete = 150 * asignacion.cant_pallets_requeridas()
            costo_total = costo_packaging + costo_flete
            metricas[caja_id] = (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
            
            # Revertimos la asignación para que no se aplique un descuento posterior
            caja.revocar_producto(producto)
        
        if criterio == "minimizar_costo_total":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][3]))
        elif criterio == "maximizar_volumen_tipo":        
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][3]))
        elif criterio == "minimizar_costo_flete":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][2]))
        elif criterio == "maximizar_utilizacion_pallet":
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][4]))
        
        caja_optima = buscar_caja_por_id(caja_id_optima)
        
        asignacion = Asignacion(producto, caja_optima)
        solucion.agregar_asignacion(asignacion)    
    
    return solucion

Armamos una función general para buscar un tipo de caja por su id.

In [21]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

Greedy (1)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con menor costo considerando el cálculo de descuentos sumando los volúmenes de producto
    Recalculo descuentos

Con los descuentos finales, vuelvo a calcular cada costo de producto

Guardamos nuevamente cajas y productos, reiniciando toda asignación previa:

In [22]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy1 = solucion_greedy(5, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy1.resumen_por_asignacion()
solucion_greedy1.resumen_general()

ValueError: min() arg is an empty sequence

In [81]:
solucion_greedy1.resumen_por_asignacion()


,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,5914abeae491eab8612131985b15994f,0.945868,1.267782,-0.3,-0.3,-0.3,-0.1,-0.2,770200.13,27619,4142850,4913050.13
1,BR0002,139211,83,1e68edd4dd3edf793634e204ec225c3b,0.949000,1.319444,-0.3,-0.3,-0.3,-0.2,-0.3,68213.39,2176,326400,394613.39
2,BR0003,172506,43,1aef9680c5b131e489d3ac61e5ace43f,0.938391,1.127481,-0.3,-0.3,-0.2,-0.2,-0.3,84527.94,1961,294150,378677.94
3,BR0004,271715,88,228ba4e5ea3c7574cbd9ce77434d4525,0.925275,1.321090,-0.2,-0.3,-0.3,-0.3,-0.3,133140.35,3774,566100,699240.35
4,BR0005,7586,106,1e68edd4dd3edf793634e204ec225c3b,0.949000,1.212279,-0.3,-0.3,-0.3,-0.2,-0.3,3717.14,119,17850,21567.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.2,-0.2,1352.89,39,5850,7202.89
423,BR0418,3942,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.2,-0.2,1931.58,55,8250,10181.58
424,BR0419,43300,54,1aef9680c5b131e489d3ac61e5ace43f,0.938391,1.244446,-0.3,-0.3,-0.2,-0.2,-0.3,24248.00,493,73950,98198.00
425,BR0420,205852,41,519a80871f454928779db9ed3e039e2b,0.933183,1.142388,-0.2,-0.3,-0.3,-0.3,-0.3,100867.48,2574,386100,486967.48


In [82]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy2 = solucion_greedy(5, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy2.resumen_por_asignacion()
solucion_greedy2.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 27
Costo packaging: 31306458.400000013
Costo flete: 131892150
Costo total: 163198608.4
Utilización de pallet promedio: 0.9234925447347336
Utilización de caja promedio: 1.2558748412249592


In [83]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy3 = solucion_greedy(5, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy3.resumen_por_asignacion()
solucion_greedy3.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 17
Costo packaging: 31121207.040000003
Costo flete: 155073300
Costo total: 186194507.04
Utilización de pallet promedio: 0.9529277809508822
Utilización de caja promedio: 1.0302309985046123


In [84]:
solucion_greedy3.resumen_por_asignacion()


,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,2060590a59a5c59ff1bb32dbffe1028b,0.970288,1.052083,0.1,-0.3,-0.3,-0.1,-0.2,770200.13,32223,4833450,5603650.13
1,BR0002,139211,83,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.271230,-0.3,-0.3,-0.3,-0.3,-0.3,68213.39,2176,326400,394613.39
2,BR0003,172506,43,67894c5a8355ca9b7e57b96362fa45f1,0.982876,0.970414,-0.2,-0.3,-0.2,0.0,-0.2,96603.36,2157,323550,420153.36
3,BR0004,271715,88,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.095214,-0.3,-0.3,-0.3,-0.3,-0.3,133140.35,4246,636900,770040.35
4,BR0005,7586,106,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.167981,-0.3,-0.3,-0.3,-0.3,-0.3,3717.14,119,17850,21567.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,ea6403e25e3a52198dad19251c207a47,0.844404,0.947362,-0.3,-0.3,-0.3,-0.2,0.1,1352.89,50,7500,8852.89
423,BR0418,3942,50,ea6403e25e3a52198dad19251c207a47,0.844404,0.947362,-0.3,-0.3,-0.3,-0.2,0.1,1931.58,71,10650,12581.58
424,BR0419,43300,54,161332db17a4c8bcba8a5651098cbe91,0.983425,0.957743,-0.3,-0.3,-0.2,-0.2,-0.3,24248.00,602,90300,114548.00
425,BR0420,205852,41,1e68edd4dd3edf793634e204ec225c3b,0.949000,0.887343,-0.2,-0.3,-0.3,-0.3,-0.3,100867.48,3217,482550,583417.48


In [85]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy4 = solucion_greedy(5, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy4.resumen_por_asignacion()
solucion_greedy4.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 8
Costo packaging: 30729293.20999999
Costo flete: 150772200
Costo total: 181501493.20999998
Utilización de pallet promedio: 0.8653388481885613
Utilización de caja promedio: 1.1722415356961153


In [86]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy5 = solucion_greedy(5, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy5.resumen_por_asignacion()
solucion_greedy5.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 24
Costo packaging: 31271719.149999965
Costo flete: 131910900
Costo total: 163182619.14999998
Utilización de pallet promedio: 0.9231614032864329
Utilización de caja promedio: 1.2561854079987966


In [87]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy6 = solucion_greedy(5, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy6.resumen_por_asignacion()
solucion_greedy6.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 27
Costo packaging: 31306458.399999984
Costo flete: 131892150
Costo total: 163198608.39999998
Utilización de pallet promedio: 0.9234925447347335
Utilización de caja promedio: 1.2558748412249596


In [88]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy7 = solucion_greedy(5, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy7.resumen_por_asignacion()
solucion_greedy7.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 17
Costo packaging: 31121207.03999998
Costo flete: 155073300
Costo total: 186194507.04
Utilización de pallet promedio: 0.9529277809508823
Utilización de caja promedio: 1.0302309985046119


In [89]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy8 = solucion_greedy(5, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy8.resumen_por_asignacion()
solucion_greedy8.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 10
Costo packaging: 30731432.619999956
Costo flete: 144668850
Costo total: 175400282.61999995
Utilización de pallet promedio: 0.9023581694264049
Utilización de caja promedio: 1.1736096055855718


#### **Greedy 2: Elección por mayor volumen**

Greedy (2)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con mayor volúmen actualmente

Con los descuentos finales, vuelvo a calcular cada costo de producto

In [90]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

In [91]:
solucion_greedy2 = Solucion(grosor=5)

for producto in productos_ordenados:
    metricas = {}  # caja_id -> (volumen_total, costo_total)
    
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        
        # volumen requerido actual (antes de asignar este producto)
        volumen_total = caja.unidades_total_requeridas()
        
        # costo simulado de asignar este producto a esta caja
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        costo_total = costo_packaging + costo_flete
        caja.revocar_producto(producto)
        
        metricas[caja_id] = (volumen_total, costo_total)
    
    # criterio: mayor volumen_total primero, y en caso de empate, menor costo_total
    caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][1]))
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy2.agregar_asignacion(asignacion)

solucion_greedy2.exportar_submmit(nombre_csv="3-greedy2_5mm")
solucion_greedy2.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,2060590a59a5c59ff1bb32dbffe1028b,0.970288,1.052083,-0.3,-0.3,-0.3,-0.1,-0.3,770200.13,32223,4833450,5603650.13
1,BR0002,139211,83,63849e2a342c002e41cdca735b7ffa35,0.887333,1.231162,-0.3,-0.3,-0.3,-0.3,-0.3,68213.39,2486,372900,441113.39
2,BR0003,172506,43,8f6cdeebd6a09f639e27ef1cbb567311,0.769843,1.252844,-0.3,-0.3,-0.3,-0.2,-0.3,84527.94,2157,323550,408077.94
3,BR0004,271715,88,63849e2a342c002e41cdca735b7ffa35,0.887333,1.060694,-0.3,-0.3,-0.3,-0.3,-0.3,133140.35,4853,727950,861090.35
4,BR0005,7586,106,63849e2a342c002e41cdca735b7ffa35,0.887333,1.131168,-0.3,-0.3,-0.3,-0.3,-0.3,3717.14,136,20400,24117.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.3,-0.3,1352.89,39,5850,7202.89
423,BR0418,3942,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.3,-0.3,1931.58,55,8250,10181.58
424,BR0419,43300,54,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.126737,-0.3,-0.3,-0.3,-0.3,-0.3,21217.00,602,90300,111517.00
425,BR0420,205852,41,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.138808,-0.3,-0.3,-0.3,-0.3,-0.3,100867.48,2860,429000,529867.48


In [92]:
solucion_greedy2.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 8
Costo packaging: 30729293.20999999
Costo flete: 150772200
Costo total: 181501493.20999998
Utilización de pallet promedio: 0.8653388481885613
Utilización de caja promedio: 1.1722415356961153


#### **Greedy 3: Ordenamiento de productos según su demanda total**

Greedy (3)

Rehacer Greedy 1 y 2 ordenando los productos de mayor a menor según la demanda.

In [93]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

# Ordenamos ahora los productos según la demanda total de mayor a menor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

In [94]:
solucion_greedy3 = Solucion(grosor=5)

for producto in productos_ordenados:
    costos_totales = {}
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        
        costo_total = costo_packaging + costo_flete
        costos_totales[caja_id] = costo_total
        
        caja.revocar_producto(producto)
    
    caja_id_optima = min(costos_totales, key=costos_totales.get)
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy3.agregar_asignacion(asignacion)
    
solucion_greedy3.exportar_submmit(nombre_csv="4-greedy3_5mm")
solucion_greedy3.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,414d4cf5c1bc8a346f7e4445ede7acbd,0.907881,1.323144,0.1,-0.3,-0.2,-0.1,0.1,772060.94,27619,4142850,4914910.94
1,BR0002,139211,83,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.271230,-0.2,-0.3,-0.3,0.1,-0.2,68213.39,2176,326400,394613.39
2,BR0003,172506,43,eb8c91d6266d086a541d9c7411a9d92f,0.960363,1.100671,-0.1,-0.2,-0.3,0.1,-0.3,84527.94,1961,294150,378677.94
3,BR0004,271715,88,228ba4e5ea3c7574cbd9ce77434d4525,0.925275,1.321090,-0.2,-0.3,-0.3,-0.3,-0.3,133140.35,3774,566100,699240.35
4,BR0005,7586,106,161332db17a4c8bcba8a5651098cbe91,0.983425,1.322476,-0.2,-0.2,-0.3,0.1,-0.2,4248.16,106,15900,20148.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.2,-0.2,1352.89,39,5850,7202.89
423,BR0418,3942,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.2,-0.2,1931.58,55,8250,10181.58
424,BR0419,43300,54,1aef9680c5b131e489d3ac61e5ace43f,0.938391,1.244446,-0.3,-0.3,-0.2,-0.2,0.0,24248.00,493,73950,98198.00
425,BR0420,205852,41,519a80871f454928779db9ed3e039e2b,0.933183,1.142388,-0.2,-0.3,-0.3,-0.3,-0.3,100867.48,2574,386100,486967.48


In [95]:
solucion_greedy3.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 24
Costo packaging: 31271719.149999965
Costo flete: 131910900
Costo total: 163182619.14999998
Utilización de pallet promedio: 0.9231614032864329
Utilización de caja promedio: 1.2561854079987966


In [96]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

In [97]:
solucion_greedy4 = Solucion(grosor=5)

for producto in productos_ordenados:
    metricas = {}  # caja_id -> (volumen_total, costo_total)
    
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        
        # volumen requerido actual (antes de asignar este producto)
        volumen_total = caja.unidades_total_requeridas()
        
        # costo simulado de asignar este producto a esta caja
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        costo_total = costo_packaging + costo_flete
        caja.revocar_producto(producto)
        
        metricas[caja_id] = (volumen_total, costo_total)
    
    # criterio: mayor volumen_total primero, y en caso de empate, menor costo_total
    caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][1]))
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy4.agregar_asignacion(asignacion)

solucion_greedy4.exportar_submmit(nombre_csv="5-greedy4_5mm")
solucion_greedy4.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,5914abeae491eab8612131985b15994f,0.945868,1.267782,-0.3,-0.3,-0.3,-0.3,-0.3,757840.37,27619,4142850,4900690.37
1,BR0002,139211,83,5914abeae491eab8612131985b15994f,0.945868,1.152155,-0.3,-0.3,-0.3,-0.3,-0.3,68213.39,2486,372900,441113.39
2,BR0003,172506,43,519a80871f454928779db9ed3e039e2b,0.933183,1.024045,-0.3,-0.3,-0.3,-0.3,-0.3,84527.94,2157,323550,408077.94
3,BR0004,271715,88,5914abeae491eab8612131985b15994f,0.945868,0.992626,-0.3,-0.3,-0.3,-0.3,-0.3,133140.35,4853,727950,861090.35
4,BR0005,7586,106,5914abeae491eab8612131985b15994f,0.945868,1.058577,-0.3,-0.3,-0.3,-0.3,-0.3,3717.14,136,20400,24117.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,1a3ffa8018de4637afcdd1e390e2602a,0.784865,1.175879,-0.3,-0.3,-0.3,-0.3,-0.3,1352.89,44,6600,7952.89
423,BR0418,3942,50,1a3ffa8018de4637afcdd1e390e2602a,0.784865,1.175879,-0.3,-0.3,-0.3,-0.3,-0.3,1931.58,62,9300,11231.58
424,BR0419,43300,54,519a80871f454928779db9ed3e039e2b,0.933183,1.130279,-0.3,-0.3,-0.3,-0.3,-0.3,21217.00,542,81300,102517.00
425,BR0420,205852,41,519a80871f454928779db9ed3e039e2b,0.933183,1.142388,-0.3,-0.3,-0.3,-0.3,-0.3,100867.48,2574,386100,486967.48


In [98]:
solucion_greedy4.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 10
Costo packaging: 30731432.619999956
Costo flete: 144668850
Costo total: 175400282.61999995
Utilización de pallet promedio: 0.9023581694264049
Utilización de caja promedio: 1.1736096055855718


Partiendo de la solucion, veo si la puedo mejorar

En cada paso:
    Cambio el tipo de caja, y me fijo si con los descuentos puedo tener un menor costo